# Uber Trip Analysis with Markov Chain

## Goal
Use Markov chain analysis to assign an importance score to each taxi zone in NYC based on the number of trips that start or end at that location.

## Workflow
1. **Data Loading & Exploration** - Load trip data and location mappingfiles
2. **Data Cleaning** - Filter outliers and invalid trips
3. **Feature Engineering** - Create zone/borough mappings and compute trip metrics
4. **Network Analysis** - Build transition matrix and calculate importance scores
5. **Validation** - Verify data consistency with geospatial data

In [392]:
import pandas as pd
import numpy as np
import string
from sodapy import Socrata
from scipy.sparse import coo_matrix
import geopandas as gpd

# ============================================================================
# Helper Functions (Define these early for use throughout the notebook)
# ============================================================================

def lookup_borough_zone(location_id, borough_dict, zone_dict):
    """Lookup borough and zone for a given location ID"""
    return borough_dict.get(location_id, 'Unknown'), zone_dict.get(location_id, 'Unknown')

def top_n_node(location_id_series, n=5):
    """Get top N most frequent locations"""
    return location_id_series.value_counts().head(n)

def graph_origin_destination(df):
    """Create a DataFrame with unique origin-destination pairs and trip counts"""
    result_df = df[['PULocationID', 'DOLocationID']].groupby(['PULocationID', 'DOLocationID']).size().reset_index(name='trip_count')
    return result_df

def calculate_degree(df_graph, origin_col):
    """Calculate the total outbound trips for each origin (outdegree)"""
    df_graph['outdegree'] = df_graph.groupby(origin_col)['trip_count'].transform('sum')
    return df_graph

def calculate_frequency(df_graph):
    """Calculate the transition probability (frequency) for each edge"""
    df_graph['frequency'] = df_graph['trip_count'] / df_graph['outdegree']
    return df_graph

In [393]:
# Load trip data and mapping files
df = pd.read_parquet('/Users/Owner1/Documents/masters/uber/yellow_tripdata_2026-01.parquet')
mapping = pd.read_csv('taxi+_zone_lookup.csv')

# Display sample data
display(df.head())
display(mapping.head())

# Check shape and columns
print(f"Trip data shape: {df.shape}")  # (3724889, 20) - 3.7 million rows and 20 columns
print(f"Mapping shape: {mapping.shape}")
print(f"Unique zones: {len(mapping['LocationID'].unique())}")

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


Trip data shape: (3724889, 20)
Mapping shape: (265, 4)
Unique zones: 265


In [394]:
# Fetch geospatial data from NYC Socrata API
client = Socrata("data.cityofnewyork.us", None)

# Fetch first 2000 records (taxi zone boundaries with geospatial data)
results = client.get("8meu-9t5y", limit=2000)
geo_df = pd.DataFrame.from_records(results)

print(f"Geospatial data fetched: {geo_df.shape[0]} records")
print(f"Columns: {geo_df.columns.tolist()}")

Geospatial data fetched: 263 records
Columns: ['the_geom', 'shape_leng', 'shape_area', 'zone', 'locationid', 'borough']


In [ ]:
# Identify location IDs present in trip data but missing from geospatial data
geo_location_ids = set(geo_df['locationid'].unique())
mapping_location_ids = set(str(id) for id in mapping['LocationID'].unique())
missing_in_geo = mapping_location_ids - geo_location_ids

borough = mapping.set_index('LocationID')['Borough'].to_dict()
zone = mapping.set_index('LocationID')['Zone'].to_dict()

print(f"Mapping zones: {len(mapping_location_ids)} | Geo zones: {len(geo_location_ids)}")
print("Zones in mapping but missing from geo data:")
for i in missing_in_geo:
    print(f"  ID {i}: {lookup_borough_zone(int(i), borough, zone)}")

In [396]:
# Create a mapping dictionary
id_remap = {
    57: 56,    # Corona
    104: 103,  # Islands
    105: 103   # Islands
}

# Apply the remap to both Pickup and Dropoff columns
df['PULocationID'] = df['PULocationID'].replace(id_remap)
df['DOLocationID'] = df['DOLocationID'].replace(id_remap)

# Drop 264 and 265 (Unknowns) since they can't be mapped geographically
df = df[~df['PULocationID'].isin([264, 265])]
df = df[~df['DOLocationID'].isin([264, 265])]

In [ ]:
# Build the origin-destination graph
df_graph = graph_origin_destination(df_filtered)

# Remove self-loops: drop trips where pickup == dropoff zone
# This prevents zones like EWR (96% self-loop) from trapping probability mass
df_graph = df_graph[df_graph['PULocationID'] != df_graph['DOLocationID']].copy()
print(f"Unique OD pairs after removing self-loops: {len(df_graph)}")

# Compute outdegree and transition probabilities
df_graph = calculate_degree(df_graph, 'PULocationID')
df_graph = calculate_frequency(df_graph)
display(df_graph.sort_values('trip_count', ascending=False).head(10))

## Step 1: Data Cleaning
Remove invalid trips (outliers and extreme values)

In [398]:
# Check for missing values in original dataset
print("Missing values per column:")
print(df.isnull().sum())  # Total missing values

Missing values per column:
VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1084720
trip_distance                  0
RatecodeID               1084720
store_and_fwd_flag       1084720
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1084720
Airport_fee              1084720
cbd_congestion_fee             0
dtype: int64


In [399]:
# visualize distance (by miles) column distribution
df['trip_distance'].describe()



count    3.699638e+06
mean     6.415502e+00
std      6.510950e+02
min      0.000000e+00
25%      1.000000e+00
50%      1.800000e+00
75%      3.700000e+00
max      2.690975e+05
Name: trip_distance, dtype: float64

The manximum trip is 270000 miles, which is likely an outlier. 
We can filter out trips that are longer than a reasonable threshold, say 100 miles, to focus on typical trips.

## Step 2: Feature Engineering
Create mappings and helper functions for zone/borough lookups

In [ ]:
# Create lookup dictionaries for zone and borough info
zone = mapping.set_index('LocationID')['Zone'].to_dict()
borough = mapping.set_index('LocationID')['Borough'].to_dict()

In [401]:
# Analyze top pickup and dropoff locations
top_5_origin = top_n_node(df_filtered['PULocationID']).reset_index()['PULocationID'].apply(lambda x: lookup_borough_zone(x, borough, zone))
top_5_destination = top_n_node(df_filtered['DOLocationID']).reset_index()['DOLocationID'].apply(lambda x: lookup_borough_zone(x, borough, zone))

print("Top 5 Pickup Locations:")
display(top_5_origin)
print("\nTop 5 Dropoff Locations:")
display(top_5_destination)

Top 5 Pickup Locations:


0           (Manhattan, Upper East Side South)
1                        (Queens, JFK Airport)
2           (Manhattan, Upper East Side North)
3                  (Manhattan, Midtown Center)
4    (Manhattan, Penn Station/Madison Sq West)
Name: PULocationID, dtype: object


Top 5 Dropoff Locations:


0        (Manhattan, Upper East Side North)
1        (Manhattan, Upper East Side South)
2               (Manhattan, Midtown Center)
3                  (Manhattan, Murray Hill)
4    (Manhattan, Times Sq/Theatre District)
Name: DOLocationID, dtype: object

## Step 3: Network Analysis
Build the origin-destination graph and compute the sparse transition matrix

In [ ]:
# Map LocationIDs to 0-based matrix indices
valid_ids = mapping['LocationID'].unique()
unique_ids = sorted(valid_ids)
id_to_idx = {loc_id: i for i, loc_id in enumerate(unique_ids)}
N = len(unique_ids)

# Map IDs to row/column indices for sparse matrix construction
df_graph['row_idx'] = df_graph['PULocationID'].map(id_to_idx).values
df_graph['col_idx'] = df_graph['DOLocationID'].map(id_to_idx).values
print(f"Matrix size: {N}x{N} | Non-zero edges: {len(df_graph)}")

In [ ]:
# Construct the sparse row-stochastic transition matrix
P = coo_matrix((df_graph['frequency'], (df_graph['row_idx'], df_graph['col_idx'])), shape=(N, N))
print(f"Transition matrix: {P.shape} | Non-zero entries: {P.nnz}")

## Step 4: Compute Importance Scores
Apply Markov chain iteration to find the steady-state distribution — each zone's score represents the long-run probability a random taxi passenger ends up there

In [405]:
# Initialize probability distribution and run Markov chain iterations
x = np.ones(N) / N  # Start with uniform distribution: 1/265 probability for each zone
P_transposed = P.T  # Transpose for efficient computation

# Run 50 iterations to reach convergence
for i in range(50):
    x = P_transposed.dot(x)


# Create results dataframe with importance scores
results = []
for i, score in enumerate(x):
    loc_id = unique_ids[i]
    b_name, z_name = lookup_borough_zone(loc_id, borough, zone)
    results.append({
        'LocationID': loc_id,
        'Borough': b_name,
        'Zone': z_name,
        'Importance_Score': score
    })

# Sort by importance score (highest first)
df_results = pd.DataFrame(results).sort_values(by='Importance_Score', ascending=False)
print("Top 10 Most Important Zones:")
display(df_results.head(10))


Top 10 Most Important Zones:


,LocationID,Borough,Zone,Importance_Score
0,1,EWR,Newark Airport,0.045314
235,236,Manhattan,Upper East Side North,0.037000
236,237,Manhattan,Upper East Side South,0.034406
160,161,Manhattan,Midtown Center,0.027896
238,239,Manhattan,Upper West Side South,0.023435
141,142,Manhattan,Lincoln Square East,0.023362
169,170,Manhattan,Murray Hill,0.023178
140,141,Manhattan,Lenox Hill West,0.022923
229,230,Manhattan,Times Sq/Theatre District,0.021535
78,79,Manhattan,East Village,0.021516


## Step 5: Dynamic Pricing Analysis
**Business Question:** Does Markov centrality predict zone revenue better than raw trip counts?

The hypothesis is that zones receiving traffic from many *different* zones (high Markov score) attract longer, more diverse trips — generating more revenue per zone — compared to zones that simply have high volume. If true, ride-share platforms should weight driver incentives and surge pricing by centrality, not just trip count.

In [ ]:
# Compute revenue and trip count per pickup zone
zone_stats = df_filtered.groupby('PULocationID').agg(
    trip_count=('total_amount', 'count'),
    total_revenue=('total_amount', 'sum'),
    avg_fare=('total_amount', 'mean')
).reset_index()

display(zone_stats.sort_values('total_revenue', ascending=False).head(10))

In [ ]:
# Merge zone stats with Markov importance scores
df_pricing = zone_stats.merge(
    df_results[['LocationID', 'Zone', 'Borough', 'Importance_Score']],
    left_on='PULocationID',
    right_on='LocationID'
).dropna()

display(df_pricing.sort_values('Importance_Score', ascending=False).head(10))

In [ ]:
from scipy.stats import pearsonr, spearmanr

# Pearson correlation (linear relationship)
r_markov, p_markov = pearsonr(df_pricing['Importance_Score'], df_pricing['total_revenue'])
r_trips, p_trips = pearsonr(df_pricing['trip_count'], df_pricing['total_revenue'])

# Spearman correlation (rank-based, more robust to outliers)
rho_markov, p_rho_markov = spearmanr(df_pricing['Importance_Score'], df_pricing['total_revenue'])
rho_trips, p_rho_trips = spearmanr(df_pricing['trip_count'], df_pricing['total_revenue'])

print("=" * 55)
print(f"{'Predictor':<25} {'Pearson r':>10} {'Spearman ρ':>12}")
print("=" * 55)
print(f"{'Markov Importance Score':<25} {r_markov:>10.4f} {rho_markov:>12.4f}")
print(f"{'Raw Trip Count':<25} {r_trips:>10.4f} {rho_trips:>12.4f}")
print("=" * 55)

winner = "Markov Score" if abs(rho_markov) > abs(rho_trips) else "Raw Trip Count"
print(f"\nBetter predictor of revenue: {winner}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Does Markov Centrality Predict Revenue Better Than Trip Count?", fontsize=14, fontweight='bold')

# Color by borough
borough_colors = {'Manhattan': '#e74c3c', 'Queens': '#3498db', 'Brooklyn': '#2ecc71',
                  'Bronx': '#f39c12', 'Staten Island': '#9b59b6', 'EWR': '#95a5a6'}
colors = df_pricing['Borough'].map(borough_colors).fillna('#bdc3c7')

# Plot 1: Revenue vs Trip Count
ax1 = axes[0]
ax1.scatter(df_pricing['trip_count'], df_pricing['total_revenue'], c=colors, alpha=0.6, edgecolors='white', linewidth=0.5, s=60)
m, b = np.polyfit(df_pricing['trip_count'], df_pricing['total_revenue'], 1)
x_line = np.linspace(df_pricing['trip_count'].min(), df_pricing['trip_count'].max(), 100)
ax1.plot(x_line, m * x_line + b, 'k--', linewidth=1.5, alpha=0.7)
ax1.set_xlabel("Raw Trip Count", fontsize=12)
ax1.set_ylabel("Total Revenue ($)", fontsize=12)
ax1.set_title(f"Revenue vs Trip Count\nPearson r = {r_trips:.4f} | Spearman ρ = {rho_trips:.4f}", fontsize=11)
ax1.ticklabel_format(style='plain', axis='both')

# Plot 2: Revenue vs Markov Score
ax2 = axes[1]
sc = ax2.scatter(df_pricing['Importance_Score'], df_pricing['total_revenue'], c=colors, alpha=0.6, edgecolors='white', linewidth=0.5, s=60)
m2, b2 = np.polyfit(df_pricing['Importance_Score'], df_pricing['total_revenue'], 1)
x_line2 = np.linspace(df_pricing['Importance_Score'].min(), df_pricing['Importance_Score'].max(), 100)
ax2.plot(x_line2, m2 * x_line2 + b2, 'k--', linewidth=1.5, alpha=0.7)
ax2.set_xlabel("Markov Importance Score", fontsize=12)
ax2.set_ylabel("Total Revenue ($)", fontsize=12)
ax2.set_title(f"Revenue vs Markov Score\nPearson r = {r_markov:.4f} | Spearman ρ = {rho_markov:.4f}", fontsize=11)
ax2.ticklabel_format(style='plain', axis='y')

# Annotate top zones on plot 2
top_zones = df_pricing.nlargest(5, 'Importance_Score')
for _, row in top_zones.iterrows():
    ax2.annotate(row['Zone'].split('/')[0], (row['Importance_Score'], row['total_revenue']),
                 textcoords="offset points", xytext=(5, 5), fontsize=7, alpha=0.85)

# Legend
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=8, label=b)
           for b, c in borough_colors.items()]
fig.legend(handles=handles, title='Borough', loc='lower center', ncol=6, bbox_to_anchor=(0.5, -0.05), fontsize=9)

plt.tight_layout()
plt.show()

## Step 6: Geospatial Validation
Merge Markov scores with NYC taxi zone polygons and render a choropleth map to visually validate the importance distribution across the city

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely import wkt

# Parse geometry and build GeoDataFrame
if isinstance(geo_df['the_geom'].iloc[0], str):
    geo_df['the_geom'] = geo_df['the_geom'].apply(wkt.loads)

geo_df = gpd.GeoDataFrame(geo_df, geometry='the_geom').set_crs(epsg=4326, inplace=True)
geo_df['locationid'] = geo_df['locationid'].astype(int)
df_results['LocationID'] = df_results['LocationID'].astype(int)

# Dissolve duplicate polygons, then merge with Markov scores
geo_df_combined = geo_df.dissolve(by='locationid', aggfunc='first').reset_index()
map_data = geo_df_combined.merge(df_results, left_on='locationid', right_on='LocationID')

# Plot choropleth
fig, ax = plt.subplots(1, 1, figsize=(15, 12))
map_data.plot(
    column='Importance_Score',
    ax=ax,
    legend=True,
    cmap='YlOrRd',
    edgecolor='black',
    linewidth=0.2,
    legend_kwds={'label': "Steady-State Probability (Importance)", 'orientation': "horizontal", 'pad': 0.05}
)
ax.set_title("NYC Taxi Zones: Markov Chain Importance Scores", fontsize=18)
ax.axis('off')
plt.show()